# Run the GST/POUNDERS seed sweep from JupyterLab

This notebook launches the same resumable runner as `run_seed_sweep.py` and streams its output live. Completed `(seed, method)` bundles are skipped automatically.

## Relationship to `GST_model.ipynb`

The experiment module preserves the active model construction, CPTPLND parameterization, GST design, weighted least-squares oracle, FPR implementation, POUNDERS/PyROL call, and D/A-optimal adaptive allocation. It excludes exploratory cells such as LM, statevector checks, NLL polish, one-off plots, and the L-optimal finite-difference metric. The configuration JSON is the authoritative record of each run.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / 'seed_sweep_experiments',
    Path.cwd() / 'GST_POUNDERS' / 'seed_sweep_experiments',
    Path('/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments'),
]
HERE = next((p.resolve() for p in candidates if (p / 'run_seed_sweep.py').exists()), None)
if HERE is None:
    raise FileNotFoundError('Could not locate seed_sweep_experiments/run_seed_sweep.py')
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
print('Experiment folder:', HERE)
print('Kernel Python:', sys.executable)

Experiment folder: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments
Kernel Python: /usr/local/bin/python


## Run controls

Start with one seed and one method. After that succeeds, change `SEED_SPEC` and `METHODS` for the full study.

In [9]:
RUN_SWEEP = True                 # Set True when ready
SEED_SPEC = '100:102'             # 100:101 means seed 100 only
# METHODS = ['adaptive_fpr']        # Full comparison: all three names below
METHODS = ['adaptive_fpr', 'fixed_fpr', 'fixed_no_fpr']
FORCE_RERUN = False               # False resumes and skips completed bundles
STOP_ON_ERROR = True

CONFIG_PATH = HERE / 'experiment_config.json'
RESULTS_DIR = HERE / 'results'

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
print('Seeds:', SEED_SPEC)
print('Methods:', METHODS)
print('Model:', config['model_kind'], config['modelpack_1q'])
print('Lengths:', config['max_lengths'])
print('Outer/inner iteration caps:', config['nfmax'], config['pyrol_max_iters'])
print('Adaptive criterion:', config['adaptive_criterion'])
print('Results:', RESULTS_DIR)

Seeds: 100:102
Methods: ['adaptive_fpr', 'fixed_fpr', 'fixed_no_fpr']
Model: 1Q smq1Q_XYZI
Lengths: [1, 2, 4, 8, 16, 32, 64]
Outer/inner iteration caps: 200 10
Adaptive criterion: D
Results: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/results


In [10]:
command = [
    sys.executable, '-u', str(HERE / 'run_seed_sweep.py'),
    '--config', str(CONFIG_PATH),
    '--seeds', SEED_SPEC,
    '--methods', ','.join(METHODS),
    '--results-dir', str(RESULTS_DIR),
]
if FORCE_RERUN:
    command.append('--force')
if STOP_ON_ERROR:
    command.append('--stop-on-error')

print('Command:')
print(' '.join(command))

if not RUN_SWEEP:
    print('Dry run only. Set RUN_SWEEP=True in the previous cell to execute.')
else:
    process = subprocess.Popen(
        command, cwd=HERE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    print('Runner return code:', return_code)
    if return_code != 0:
        raise RuntimeError(f'Seed sweep exited with code {return_code}; inspect results/**/failed.json')

Command:
/usr/local/bin/python -u /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/run_seed_sweep.py --config /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/experiment_config.json --seeds 100:102 --methods adaptive_fpr,fixed_fpr,fixed_no_fpr --results-dir /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/results --force --stop-on-error
Data seeds: [100, 101]
Methods: ['adaptive_fpr', 'fixed_fpr', 'fixed_no_fpr']
Results: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/results
Truth model: fixed seed 100
RUN seed=100 method=adaptive_fpr
[POUDERS] Beginning gradient-based optimization.
[POUDERS] Initial residual/Jacobian evaluation took 1.16 seconds.
/usr/local/lib/python3.11/site-packages/pygsti/forwardsims/mapforwardsim.py:732: UserWarning: Generating dense process matrix representations of circuits or gates 
can be inefficient and should be avoided for the purposes of forward 
simulation/calculation of circuit outcome probability distributions 
when using the Map

## Inspect completed runs

In [7]:
from analysis_utils import load_results, method_summary

runs = load_results(RESULTS_DIR)
print('Completed runs:', len(runs))
display(runs)
if not runs.empty:
    display(method_summary(runs))

Completed runs: 1


,accounted_revealed_shots,data_seed,dof,flag,logl_max,logl_model,max_probability,max_shots_per_circuit,mean_gate_average_infidelity_to_ideal,mean_gate_average_infidelity_to_truth,...,num_probability_below_zero,num_residuals,physical_precomputed_shots,reduced_chi_square,revealed_circuits,total_circuits,truth_seed,two_delta_logl,weighted_least_squares_objective,xkin
0,483706,100,1905,544.631044,-634623.982917,-635549.685717,0.999172,9998,0.004997,0.000036,...,0,3836,1117706,0.971391,650,1918,100,1851.405601,3700.998824,199


,method,num_seeds,mean_gate_entanglement_infidelity_to_truth_median,mean_gate_entanglement_infidelity_to_truth_q25,mean_gate_entanglement_infidelity_to_truth_q75,mean_spam_vector_l2_error_to_truth_median,mean_spam_vector_l2_error_to_truth_q25,mean_spam_vector_l2_error_to_truth_q75,accounted_revealed_shots_median,accounted_revealed_shots_q25,accounted_revealed_shots_q75,reduced_chi_square_median,reduced_chi_square_q25,reduced_chi_square_q75,n_sigma_median,n_sigma_q25,n_sigma_q75
0,adaptive_fpr,1,0.000054,0.000054,0.000054,0.0031,0.0031,0.0031,483706.0,483706.0,483706.0,0.971391,0.971391,0.971391,-0.868274,-0.868274,-0.868274


For the complete plots and paired-seed tables, open `analyze_seed_sweep.ipynb` after the sweep finishes.